In [31]:
import torch
import re
from collections import defaultdict



In [32]:
text = open("train.txt", "r", encoding="utf-8").read()

unique_chars = sorted(list(set(text)))
vocab_size = len(unique_chars)


print(''.join(unique_chars))
print(f"\nvocab_Size:{vocab_size}")

# unique_chars.extend(['4','6','7','9','0'])
# Adding remaining numbers 

def word_tokenize(text):
    """Each word is a token, but also each special character, spaces arent ocnsidered."""
    # Split on spaces but keep punctuation as separate tokens
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens
word_tokens = word_tokenize(text)


print(f"\nFirst 50 WORD tokens:\n {word_tokens[:50]}")




 !"&'(),-./12358:;?ABCDEFGHIJKLMNOPQRSTUVWYabcdefghijklmnopqrstuvwxyzé

vocab_Size:71

First 50 WORD tokens:
 ['A', 'SCANDAL', 'IN', 'BOHEMIA', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'Chapter', '1', 'Chapter', '2', 'Chapter', '3', 'CHAPTER', 'I', 'To', 'Sherlock', 'Holmes', 'she', 'is', 'always', 'the', 'woman', '.', 'I', 'have', 'seldom', 'heard', 'him', 'mention', 'her', 'under', 'any', 'other', 'name', '.', 'In', 'his', 'eyes', 'she', 'eclipses', 'and', 'predominates', 'the', 'whole', 'of', 'her']


In [33]:
def split_words_with_sep(text):
    for m in re.finditer(r'(\S+)(\s*)', text):
        word, sep = m.groups()
        if not word:
            continue
        marker = '<para>' if sep.count('\n') >= 2 else '<end>'
        yield word, marker

def get_vocab(text):
    vocab = defaultdict(int)
    for word, marker in split_words_with_sep(text):
        vocab[' '.join(list(word)) + ' ' + marker] += 1
    return vocab

def get_stats(vocab):
    """Count frequency of every adjacent pair and stores in dict pairs"""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_mostfrequent(best_pair, vocab):
    """Merge the most frequent pair across all words, variable- pair is the most freq adjacent pair"""
    new_vocab = {}
    to_replace = ' '.join(best_pair)
    replacement = ''.join(best_pair)
    for word in vocab:
        new_word = word.replace(to_replace, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab


def train_bpe(text, n_merges):
    vocab = get_vocab(text)
    merges = []

    for i in range(n_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        # merge the most frequent pair
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_mostfrequent(best_pair, vocab)
        merges.append(best_pair)

    return vocab, merges

print("\n\n")
bpe_vocab, merges = train_bpe(text, n_merges=100)

# Build final token list from BPE vocab
bpe_tokens = set()
for word in bpe_vocab:
    for token in word.split():
        bpe_tokens.add(token)

bpe_tokens = sorted(list(bpe_tokens))
print(f"BPE vocab size: {len(bpe_tokens)}")
print(f"First 20 tokens: {bpe_tokens[:20]}")

stoi = {token: i for i, token in enumerate(bpe_tokens)}
itos = {i: token for i, token in enumerate(bpe_tokens)}

bpe_vocab_size=len(bpe_tokens)
#string ot integer and integer to string

def encode(text):
    token_ids = []
    for word, marker in split_words_with_sep(text):
        symbols = list(word) + [marker]
        for pair in merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                    symbols = symbols[:i] + [''.join(pair)] + symbols[i+2:]
                else:
                    i += 1
        for token in symbols:
            if token in stoi:
                token_ids.append(stoi[token])
    return token_ids

encoded = encode(text)

def decode(token_ids):
    tokens = [itos[i] for i in token_ids]
    text = ''.join(tokens)
    text = text.replace('<para>', '\n\n').replace('<end>', ' ')
    return text.strip()

n = int(0.9 * len(encoded))
train = encoded[:n]
test = encoded[n:]


train = torch.tensor(train, dtype=torch.long)
test = torch.tensor(test, dtype=torch.long)

print(f"Train tokens: {len(train)}, Test tokens: {len(test)}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")






BPE vocab size: 454
First 20 tokens: ['!', '"', '"<end>', '"<para>', '&', "'", '(', ')', ',', ',<end>', '-', '.', '."<para>', '.<end>', '.<para>', '/', '1', '2', '3', '5']
Train tokens: 24120, Test tokens: 2680
Device: CPU


In [34]:
torch.manual_seed(10)
block_size = 8   # context length 
batch_size = 4   #no of sampleing chunks in one forward pass

def get_batch(data):
    #random start positions
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # x is the input context, y is the target (shifted by 1)
    x = torch.stack([data[i : i+block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    
    return x, y

xb, yb = get_batch(train)
print(xb.shape)
print(yb.shape,"\n\n")

print("INPUTS",xb)
print("OUTPUTS:",yb)

    

torch.Size([4, 8])
torch.Size([4, 8]) 


INPUTS tensor([[277, 213, 310, 268, 230, 152, 157, 267],
        [ 90, 152, 445, 383, 314,  51, 295, 150],
        [446,  73,  23, 123, 314, 303, 180, 186],
        [250, 410, 150, 275,   8,   2,  35, 316]])
OUTPUTS: tensor([[213, 310, 268, 230, 152, 157, 267, 113],
        [152, 445, 383, 314,  51, 295, 150, 216],
        [ 73,  23, 123, 314, 303, 180, 186, 265],
        [410, 150, 275,   8,   2,  35, 316, 265]])


In [35]:
torch.manual_seed(10)
import torch.nn as nn  

import torch.nn.functional as F

class BigramLM(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        #embedding table. or lookup table.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self,index,targets=None):
        logits=self.token_embedding_table(index)
        #cross entropy needs a B C T not B T C matrix so we reshape.
        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C) #stretches the original 4x8 array into 1d
            targets=targets.view(B*T) #stretches the original 4x8 array into 1d
            loss=F.cross_entropy(logits,targets)

        return logits,loss

    def generate(self, index, max_new_tokens):
        #index is B x T array of indices in the current context,.
        for i in range(max_new_tokens):

            logits, loss = self.forward(index,None) 
            # gives B T C , we need last one.,
            logits = logits[:, -1, :]  # shape (B, C)

            probs = F.softmax(logits, dim=-1)  # shape (B, C)

            next_token = torch.multinomial(probs, num_samples=1)  # shape (B, 1)

            #pppend ,repeat
            index = torch.cat([index, next_token], dim=1) 

        return index
    
model=BigramLM(bpe_vocab_size)
logits,loss=model(xb,yb)
print(logits.shape)
print(loss)   #lideally loss should be -ln(1/bpe_vocab_size)
#output is a PREDICTION score for EACH element in the torch.stack 2d array, for each unique token, it gives a UN-NORMALISED probability that thats the next token


torch.Size([32, 454])
tensor(6.8289, grad_fn=<NllLossBackward0>)


In [36]:
context = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(context, max_new_tokens=20)
print(decode(generated[0].tolist()))   # random because model isnt trained, and generate function puts random shit

!y ithllowwinwhint a t lingalBsharroll;WK&her ay


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32
best_loss = 2.8533  

for i in range(1000):
    xb, yb = get_batch(train)
    logits, loss = model(xb, yb)    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if loss.item() < best_loss:
        best_loss = loss.item()
        torch.save(model.state_dict(), 'bigram_weights.pth')

    if i%250==0:
        print(f"loss {loss.item():.4f}, min is {best_loss:.4f}")

print(f"Training done. Best loss: {best_loss:.4f}")

loss 3.2279, min is 2.8533
loss 3.0423, min is 2.8533
loss 3.2260, min is 2.8533
loss 3.1209, min is 2.8533
loss 3.0631, min is 2.8533
Training done. Best loss: 2.8533


In [41]:
model.load_state_dict(torch.load('bigram_weights.pth'))

context = torch.tensor([encode("Holmes")], dtype=torch.long)

generated = model.generate(context, max_new_tokens=500)
print(decode(generated[0].tolist()))

Holmes of now is ce be in th ther le, was a le ped G"

"An, texcing as chan," mes of youasthenshe orme as to eprod,'Egagentrdever lihere where has olitiit had ritlim," assmbonst."

"Holiocart, be a then mby guree marly pullave ter burnugar osicisailad leno 5besarmor, and hore becassoincerning dule tion the Selient evanged. Ano nothaincy pen ted, cole en. It been s. Hown plers ontapsuch inlallend ceely ichas osulhe.

"Qthatatheerest le," s, ies," I rosen will you mbit?" shed malls dgemadjWhow selsly do is trinju, hery hat is the nd bloseckedroom wonouthre. He bstane you mader.

"this an ant. I locen, to an to his der She befricightittest umaris."

"Wedlasurured sen ses asesaring xcopapplimordo it had dat sever his own se, would helf in itork fit I thry could not been would oll, and ged. Pokmurnow cenghad upon hat man im mokin feenton. It her ind Hold.

"I ce in th and res; to op ther probsidly inture not mit com
